# Week 3, day 4 (morning) — Worksheet 04 SOLUTIONS: star, snowflake and galaxy

Executed in the lab image. Every quoted number is what it actually printed.

Question 7 is the one that costs people money in production. Two fact tables and
a shared dimension look like an invitation to join, and the result is not wrong
by a little.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 04 — Star, snowflake and galaxy. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr = load("enrollment")
tx = load("transaction")
crs, prg, cat = load("course"), load("program"), load("category")

print("enrollment: ", enr.shape)
print("transaction:", tx.shape)
print("course/program/category:", crs.shape, prg.shape, cat.shape)

PART A — the same hierarchy, two shapes

### Question 1

Build the **star** version of the course dimension: one table with `course_id`, `course_name`, `credit_hours`, `program_name` and `program_category`, flattened. Print its shape and first three rows.
> **NOTE:** apply slide 36's rule to the missing category — `fillna("Unknown")` — or 316 enrollments will disappear later.

In [ ]:
dim_course_star = (
    crs[["course_id", "course_name", "hours", "program_id"]]
    .merge(prg[["program_id", "program_name", "category_id"]], on="program_id")
    .merge(cat[["category_id", "category_name"]], on="category_id", how="left")
    .rename(columns={"hours": "credit_hours",
                     "category_name": "program_category"}))
dim_course_star["program_category"] = (
    dim_course_star["program_category"].fillna("Unknown"))
dim_course_star = dim_course_star[
    ["course_id", "course_name", "credit_hours",
     "program_name", "program_category"]]
print("dim_course (star):", dim_course_star.shape)
print()
print(dim_course_star.head(3).to_string(index=False))

```
dim_course (star): (24, 5)

 course_id                  course_name  credit_hours                 program_name program_category
       201       SQL for Data Engineers            42     Applied Data Engineering Data Engineering
       202 Data Modeling and ETL Design            30        Analytics Engineering Data Engineering
       203  Python for Data Engineering            42 Machine Learning Foundations     Data Science
```

Twenty-four rows, five columns, three levels of hierarchy flattened into one
table. A fact row carrying `course_id` can now reach `program_category` in a
single join.

This is what slide 18 means by *"denormalized dimension tables"*. The program
name and category are **copied onto every course row** — `Data Engineering`
appears six times in a 24-row table — and that duplication is the deliberate
price of the single join.

Note where `program_category` came from: `category.category_name`, renamed. Slide
29 names the column `program_category` on `dim_program`, not `category_name`, and
the rename is not cosmetic. In the source it is a category *of a program*; in the
model it is an attribute *of a course*. Dimension columns get named for what they
mean to the person reading the report, not for the source table they came from.

The `fillna("Unknown")` matters more than it looks. Without it the 316
enrollments in `Foundations Bootcamp` carry a null category, and worksheet 01
question 4 showed exactly what happens next: `groupby` drops them and the report
is short by 13% with no error. Question 3 confirms they survive.

### Question 2

Build the **snowflake** version: three tables — `dim_course_sf`, `dim_program_sf`, `dim_category_sf` — each holding its own attributes and a key to the next level up. Print the shape of each and the total rows stored.

In [ ]:
dim_category_sf = cat[["category_id", "category_name"]].copy()
dim_category_sf["category_name"] = dim_category_sf["category_name"].fillna("Unknown")
dim_program_sf = prg[["program_id", "program_name", "category_id"]].copy()
dim_course_sf = (crs[["course_id", "course_name", "hours", "program_id"]]
                 .rename(columns={"hours": "credit_hours"}))

for name, df in [("dim_course_sf", dim_course_sf),
                 ("dim_program_sf", dim_program_sf),
                 ("dim_category_sf", dim_category_sf)]:
    print("  %-18s %s  %s" % (name, df.shape, list(df.columns)))
print()
print("snowflake, total rows stored:",
      len(dim_course_sf) + len(dim_program_sf) + len(dim_category_sf))

```
  dim_course_sf      (24, 4)  ['course_id', 'course_name', 'credit_hours', 'program_id']
  dim_program_sf     (8, 3)   ['program_id', 'program_name', 'category_id']
  dim_category_sf    (5, 2)   ['category_id', 'category_name']

snowflake, total rows stored: 37
```

Three tables, 37 rows. Every value stored exactly once: five category names, not
24 copies of five category names.

This is the shape slide 19 describes — *"a variation of a star schema where some
dimensions are split into related sub-dimension tables"* — and it is simply the
source model's own normalisation, preserved rather than flattened.

The structure is a chain: `dim_course_sf` holds `program_id`, `dim_program_sf`
holds `category_id`, and `dim_category_sf` is the end of it. To get from a fact
row to a category name you traverse all three.

Slide 19 lists three conditions for choosing this:

- *dimension hierarchies are large or reusable*
- *shared reference tables need to be managed separately*
- *maintenance concerns outweigh query simplicity*

None of them apply here, and the row counts say why: **37 rows against 24**. The
saving is 13 rows. Questions 4 and 5 measure both sides of that trade properly,
but the headline is already visible — at this scale the normalisation is buying
almost nothing.

Where it does pay is scale and reuse. A product dimension with 2 million rows and
a 40-character category name is storing 80 MB of repeated text; a category table
used by four different dimensions genuinely does want to be one table. Neither is
true of five education categories.

### Question 3

Answer *"enrollments per program category"* from each shape. Count the joins each needs (including the fact table) and confirm both give the same answer.

In [ ]:
dim_course_star = (
    crs[["course_id", "hours", "program_id"]]
    .merge(prg[["program_id", "program_name", "category_id"]], on="program_id")
    .merge(cat[["category_id", "category_name"]], on="category_id", how="left"))
dim_course_star["category_name"] = dim_course_star["category_name"].fillna("Unknown")

star = (enr.merge(dim_course_star[["course_id", "category_name"]], on="course_id")
           .groupby("category_name").size().rename("enrollments"))
print("STAR -- 1 join (fact -> dim_course)")
print(star.sort_values(ascending=False).to_string())
print()

cat_sf = cat[["category_id", "category_name"]].copy()
cat_sf["category_name"] = cat_sf["category_name"].fillna("Unknown")
snow = (enr.merge(crs[["course_id", "program_id"]], on="course_id")
           .merge(prg[["program_id", "category_id"]], on="program_id")
           .merge(cat_sf, on="category_id")
           .groupby("category_name").size().rename("enrollments"))
print("SNOWFLAKE -- 3 joins (fact -> course -> program -> category)")
print(snow.sort_values(ascending=False).to_string())
print()
print("identical:", star.sort_index().equals(snow.sort_index()))

```
STAR -- 1 join (fact -> dim_course)
category_name
Cloud Computing     616
Data Science        603
Data Engineering    587
Unknown             316
Cybersecurity       278

SNOWFLAKE -- 3 joins (fact -> course -> program -> category)
category_name
Cloud Computing     616
Data Science        603
Data Engineering    587
Unknown             316
Cybersecurity       278

identical: True
```

Same five rows, same five numbers. **One join against three.**

That is the entire query-side argument, and it is worth being precise about what
it costs. Not correctness — the answers are identical, and they will always be
identical, because both shapes encode the same relationships. What differs is:

**What the analyst has to know.** The star version requires knowing that
`dim_course` has a `program_category` column. The snowflake version requires
knowing the chain — course to program to category — and getting it in the right
order. That is the cost slide 21 calls *"slightly harder queries"*, and it is
paid by every person who writes a query, forever, rather than once by the person
who built the model.

**What the BI tool can do unaided.** Most tools handle a star natively: point at
the fact table, pick dimension attributes, done. Multi-hop snowflake chains often
need the relationships defined in the semantic layer first, and some tools
degrade to worse query plans across them.

**Note `Unknown` is now in the output**, at 316 enrollments — 13% of the total,
the fourth largest category. In worksheet 01 those rows were silently absent from
this exact report. One `fillna` at load time turned an invisible hole into a
visible line item that somebody can now ask about.

That is the pattern this whole day is about: a fix applied once, at load, in the
model — rather than in every query, by everyone, forever, if they remember.

### Question 4

Measure slide 21's *"may duplicate dimension details"*. In the star dimension, count how many rows repeat each `program_category` value and total the characters stored; do the same for the snowflake's category table.

In [ ]:
dim_course_star = (
    crs[["course_id", "program_id"]]
    .merge(prg[["program_id", "program_name", "category_id"]], on="program_id")
    .merge(cat[["category_id", "category_name"]], on="category_id", how="left"))
dim_course_star["category_name"] = dim_course_star["category_name"].fillna("Unknown")

print("times each category is stored in the STAR dimension:")
print(dim_course_star.category_name.value_counts().rename("rows").to_string())
print()
star_chars = dim_course_star.category_name.str.len().sum()
sf_chars = (cat[["category_name"]].fillna("Unknown")
            .category_name.str.len().sum())
print("characters, star dimension:      %5d over %d rows"
      % (star_chars, len(dim_course_star)))
print("characters, snowflake category:  %5d over %d rows" % (sf_chars, len(cat)))
print("ratio: %.1fx" % (star_chars / sf_chars))

```
times each category is stored in the STAR dimension:
category_name
Data Engineering    6
Data Science        6
Cloud Computing     6
Cybersecurity       3
Unknown             3

characters, star dimension:        318 over 24 rows
characters, snowflake category:     63 over 5 rows
ratio: 5.0x
```

Five times the characters. Slide 21's *"may duplicate dimension details"*,
measured — and the honest reading is that **it does not matter here.**

318 bytes against 63. On a dimension of 24 rows this is not a consideration; it
is a rounding error on a rounding error. If storage were the only argument for
snowflaking, nobody would ever snowflake anything.

Two reasons the number is this small, and both generalise:

**Dimensions are small by nature.** They describe things, and businesses have
fewer things than events. This one has 24 rows against a 2,400-row fact table —
a hundred to one, and that ratio usually gets worse over time, since facts
accumulate and courses do not.

**Columnar stores dictionary-encode repeated values anyway.** Snowflake,
BigQuery and Parquet all replace 24 category strings with 24 small integers and
one dictionary of five entries. The 5.0x measured here is close to a worst case
for row-based storage, and closer to 1x on the platforms most warehouses actually
run on.

So if duplication is not the real cost, what is? **Consistency.** The same five
categories now exist as 24 copies, and 24 copies are 24 chances to disagree.
Nothing in the star dimension prevents row 7 from saying `Data Engineering` and
row 12 from saying `Data engineering` — two categories in every report, splitting
the totals.

That is the argument for snowflaking, and it is not about bytes. Question 5
measures it.

### Question 5

Now measure the other side. Rename `"Cloud Computing"` to `"Cloud and Platform Engineering"` in each shape, and print how many rows each update touches.
> **NOTE:** this is the argument *for* snowflaking, and it is the mirror image of question 4.

In [ ]:
dim_course_star = (
    crs[["course_id", "program_id"]]
    .merge(prg[["program_id", "category_id"]], on="program_id")
    .merge(cat[["category_id", "category_name"]], on="category_id", how="left"))
dim_course_star["category_name"] = dim_course_star["category_name"].fillna("Unknown")

OLD, NEW = "Cloud Computing", "Cloud and Platform Engineering"
star_rows = int((dim_course_star.category_name == OLD).sum())
sf_rows = int((cat.category_name == OLD).sum())
print("rows to UPDATE to rename one category:")
print("  star dimension:      %3d" % star_rows)
print("  snowflake category:  %3d" % sf_rows)
print()
print("rows that must ALL be changed consistently, or reports split:",
      star_rows if star_rows > sf_rows else sf_rows)
print()
print("and if the star update half-fails, grouping by category gives:")
partial = dim_course_star.category_name.replace({OLD: NEW}).head(3).tolist()
print("  a mix of %r and %r" % (OLD, NEW))

```
rows to UPDATE to rename one category:
  star dimension:        6
  snowflake category:    1

rows that must ALL be changed consistently, or reports split: 6

and if the star update half-fails, grouping by category gives:
  a mix of 'Cloud Computing' and 'Cloud and Platform Engineering'
```

Six rows against one. This is the real cost of denormalisation, and it is not the
storage from question 4 — it is the **update anomaly**.

The failure mode is specific. The star update is six separate row modifications
that must all succeed. If four succeed and two do not — a transaction that
half-committed, a batch job killed mid-run, a `WHERE` clause that missed two rows
— you now have two category values where the business has one. Every report
grouped by category shows both, each with part of the total, and both look like
real categories. Nothing errors. The only symptom is that a number somebody knew
went down.

In the snowflake shape that cannot happen. There is one row holding the name, so
there is nothing for it to disagree with. **Normalisation makes the inconsistent
state unrepresentable**, which is a much stronger guarantee than being careful.

So the two questions frame the real trade-off, and neither is about performance:

| | star | snowflake |
|---|---|---|
| joins to reach a category | 1 | 3 |
| storage for category names | 318 chars | 63 chars |
| rows to update on a rename | 6 | 1 |
| can the copies disagree? | **yes** | no |

Slide 21's guidance — *"start with a star schema unless there is a clear reason
to snowflake"* — holds because in a warehouse the star's weakness is largely
defused. Dimensions are **rebuilt by the ELT**, not updated in place by users; a
rename in the source propagates to all six rows on the next load, from one place,
by one process. The anomaly is a risk when humans edit dimension tables by hand,
and that is not how a warehouse should work.

Which reframes the answer usefully: **choose the star, and make the load
authoritative.** If someone is running manual `UPDATE`s against a dimension
table, the schema shape is not your biggest problem.

PART B — galaxy: two facts, shared dimensions

### Question 6

Slide 29's note says a separate `fact_payment` turns this into a galaxy schema. Build a minimal one at payment grain — `trans_id`, `enrl_id`, `course_id`, `payment_amount` — and print both fact tables' shapes and grains, plus the dimensions they share.

In [ ]:
fact_enrollment = enr[["enrl_id", "course_id", "cohort_id", "stu_id"]].copy()
fact_enrollment["enrollment_count"] = 1

fact_payment = (tx[["trans_id", "enrl_id", "pymt_type_id", "payment_amount"]]
                .merge(enr[["enrl_id", "course_id", "cohort_id"]],
                       on="enrl_id", how="left"))

print("fact_enrollment %s  grain: one row per enrollment" % (fact_enrollment.shape,))
print("fact_payment    %s  grain: one row per payment" % (fact_payment.shape,))
print()
shared = sorted(set(fact_enrollment.columns) & set(fact_payment.columns)
                - {"enrollment_count"})
print("keys present in both:", shared)
print()
print("dimensions shared: dim_course, dim_cohort  (dim_date too, via each date)")
print("dimensions unique to fact_payment: dim_payment_type")

```
fact_enrollment (2400, 5)  grain: one row per enrollment
fact_payment    (4856, 6)  grain: one row per payment

keys present in both: ['cohort_id', 'course_id', 'enrl_id']

dimensions shared: dim_course, dim_cohort  (dim_date too, via each date)
dimensions unique to fact_payment: dim_payment_type
```

Two fact tables, **two different grains**, sharing three keys. That is a galaxy
schema — slide 20's *"fact constellation"*.

The reason to build it is the thing worksheet 02 question 9 proved you must not
do: put both grains in one table. Payments and enrollments are different events,
measured differently, counted differently. Forcing them into one fact table makes
every measure in it wrong. Giving them one table each, joined to the same
dimensions, keeps both correct.

This is exactly what slide 29's note anticipates: *"For detailed payment
analysis, add a separate fact_payment table. This would turn the model into a
galaxy schema."*

What the galaxy buys is **conformed dimensions**. `dim_course` is one table, used
by both facts, so "course 214" means the same thing in an enrollment report and a
payment report. Filter both to `program_category = 'Data Engineering'` and the
filter means the same thing to each. Without conformed dimensions, comparing the
two processes requires reconciling two definitions of a course first, and that
reconciliation is where most cross-domain reporting projects die.

`dim_payment_type` attaches only to `fact_payment`, which is normal and fine. A
galaxy does not require every dimension to be shared — only that the shared ones
are genuinely the same table.

The cost is slide 20's *"more complex model design"*, and question 7 shows the
specific shape that complexity takes.

### Question 7

Now do the thing a galaxy schema invites. Join `fact_enrollment` to `fact_payment` on their shared `course_id` and print the row count, along with `SUM(enrollment_count)` and `SUM(payment_amount)` before and after.
> **NOTE:** both tables have many rows per `course_id`. Predict the row count before you run it.

In [ ]:
fact_enrollment = enr[["enrl_id", "course_id"]].copy()
fact_enrollment["enrollment_count"] = 1
fact_payment = (tx[["trans_id", "enrl_id", "payment_amount"]]
                .merge(enr[["enrl_id", "course_id"]], on="enrl_id", how="left"))

j = fact_enrollment.merge(fact_payment, on="course_id", suffixes=("_e", "_p"))
print("fact_enrollment rows: %8d" % len(fact_enrollment))
print("fact_payment rows:    %8d" % len(fact_payment))
print("joined rows:          %8d" % len(j))
print("blow-up factor: %.0fx the larger table" % (len(j) / len(fact_payment)))
print()
print("SUM(enrollment_count)  before %10d   after %12d"
      % (fact_enrollment.enrollment_count.sum(), j.enrollment_count.sum()))
print("SUM(payment_amount)    before %10.2f   after %15.2f"
      % (fact_payment.payment_amount.sum(), j.payment_amount.sum()))

```
fact_enrollment rows:     2400
fact_payment rows:        4856
joined rows:            491688
blow-up factor: 101x the larger table

SUM(enrollment_count)  before       2400   after       491688
SUM(payment_amount)    before 8953821.53   after    907818975.86
```

**Half a million rows out of 2,400 and 4,856. Payment revenue of 8,953,821.53
becomes 907,818,975.86 -- 101 times too large.**

The mechanism is a cartesian product within each key group. `course_id` is not
unique in *either* table — course 204 has 120 enrollments and roughly 240
payments — so joining on it pairs every enrollment of that course with every
payment of that course. 120 x 240 is 28,800 rows where there should be a few
hundred, and that happens 24 times.

This is the **chasm trap**: two many-to-one relationships pointing at the same
dimension, joined *through* it as if they related to each other. They do not. An
enrollment and a payment are related through `enrl_id`, not through the fact that
both involve course 204.

What makes it dangerous in a galaxy schema specifically is that the model
*invites* the query. Two fact tables, a shared key column, the same name in
both — joining them is the obvious move, and no tool stops you. Worse, a drag-and-drop
BI tool will construct exactly this join when a user puts enrollments and
payments on one chart.

**The symptom to recognise:** a total that is wrong by a large, weird, unstable
multiple. Not 2x — 101x here, and the factor changes as the data grows, because
it depends on the product of the two group sizes. If a number jumps by two orders
of magnitude when you add a table to a query, you have fanned out.

Note that `SUM(enrollment_count)` is also destroyed — 491,688 "enrollments" — so
both measures are ruined, not just the one from the fanned-out side. In a join
like this there is no safe column.

Question 8 is the fix, and it is not complicated.

### Question 8

Do it correctly instead. Aggregate each fact table to `course_id` **separately**, then join the two summaries. Print the result for five courses and confirm both totals survive.
> **NOTE:** this is the standard fix — aggregate to a common grain first, join second. It is sometimes called avoiding the *chasm trap*.

In [ ]:
fact_enrollment = enr[["enrl_id", "course_id"]].copy()
fact_enrollment["enrollment_count"] = 1
fact_payment = (tx[["trans_id", "enrl_id", "payment_amount"]]
                .merge(enr[["enrl_id", "course_id"]], on="enrl_id", how="left"))

a = fact_enrollment.groupby("course_id").enrollment_count.sum()
b = fact_payment.groupby("course_id").payment_amount.sum()
summary = pd.concat([a, b], axis=1).fillna(0)
print(summary.head(5).round(2).to_string())
print()
print("courses:                    ", len(summary))
print("SUM(enrollment_count)  %10d  (true %d)"
      % (summary.enrollment_count.sum(), len(fact_enrollment)))
print("SUM(payment_amount) %13.2f  (true %.2f)"
      % (summary.payment_amount.sum(), fact_payment.payment_amount.sum()))

```
           enrollment_count  payment_amount
course_id
201                      98       340569.99
202                      90       349972.94
203                     113       429946.02
204                     120       435893.05
205                     104       390086.32

courses:                     24
SUM(enrollment_count)        2400  (true 2400)
SUM(payment_amount)    8953821.53  (true 8953821.53)
```

Both totals exact. 2,400 enrollments and 8,953,821.53 in payments, in one table
with one row per course.

The fix is a reordering, not a repair: **aggregate each fact table to the common
grain first, join the summaries second.** Each `SUM` runs over its own table, at
its own grain, before anything is joined — so there is nothing to fan out. By the
time the join happens both sides have exactly one row per `course_id`, and a
one-to-one join cannot multiply anything.

In SQL this is the pattern with two subqueries, or two CTEs:

```sql
WITH e AS (SELECT course_id, SUM(enrollment_count) AS enrollments
           FROM fact_enrollment GROUP BY course_id),
     p AS (SELECT course_id, SUM(payment_amount) AS paid
           FROM fact_payment GROUP BY course_id)
SELECT e.course_id, e.enrollments, p.paid
FROM e FULL OUTER JOIN p USING (course_id);
```

Three details worth keeping.

**`FULL OUTER`, not inner.** A course with enrollments and no payments belongs in
this report as a zero, not as a missing row — the same argument as worksheet 01
question 7. The `fillna(0)` in the solution does that job here.

**The join grain must be a real shared dimension.** `course_id` works because
both facts genuinely have one. Joining on something only one of them has produces
nonsense of a different kind.

**This is what BI semantic layers do for you.** Looker, dbt's metrics layer and
Power BI's model relationships all implement some version of "aggregate then
join" precisely because hand-written cross-fact joins get this wrong so
reliably. If your platform has one, define the fact tables in it and let it plan
the query. If it does not, the CTE pattern above is the discipline.

**The rule to remember:** never join two fact tables directly. Aggregate each to
a conformed grain, then join.

PART C — putting numbers on slide 21

### Question 9

Fill in slide 21's comparison with figures from this data: for star and snowflake, the number of dimension tables, total rows stored, and joins needed to reach `program_category` from the fact table.

In [ ]:
dim_course_star = (
    crs[["course_id", "program_id"]]
    .merge(prg[["program_id", "category_id"]], on="program_id")
    .merge(cat[["category_id", "category_name"]], on="category_id", how="left"))

rows = [
    ("star", 1, len(dim_course_star), 1),
    ("snowflake", 3, len(crs) + len(prg) + len(cat), 3),
]
print("%-11s %7s %12s %8s" % ("shape", "tables", "rows stored", "joins"))
for name, t, r, j in rows:
    print("%-11s %7d %12d %8d" % (name, t, r, j))
print()
print("galaxy adds a second FACT table, not a dimension:")
print("  fact_enrollment %d rows + fact_payment %d rows, sharing dim_course"
      % (len(enr), len(tx)))

```
shape        tables  rows stored    joins
star              1           24        1
snowflake         3           37        3

galaxy adds a second FACT table, not a dimension:
  fact_enrollment 2400 rows + fact_payment 4856 rows, sharing dim_course
```

Slide 21's table, with this dataset's numbers in it:

| Model | Tables | Rows stored | Joins to a category | Main trade-off, measured |
|---|---|---|---|---|
| Star | 1 | 24 | 1 | 6 rows to update on a rename; copies can disagree |
| Snowflake | 3 | 37 | 3 | 3 joins instead of 1, and the chain must be known |
| Galaxy | — | — | — | a cross-fact join gives **101x** the right answer |

The last line is the one to keep. The star/snowflake decision here is genuinely
minor — 13 rows of storage against two extra joins, on a 24-row dimension. Either
choice produces a working model, and slide 21's advice to default to star is
sound because the star's weakness is defused by rebuilding dimensions from the
load rather than editing them.

The galaxy row is different in kind. It is not a trade-off between two acceptable
designs; it is a correctness hazard that the design introduces. Adding
`fact_payment` is the right modelling decision — the alternative is mixing grains
in one table, which worksheet 02 question 9 showed is worse — but it creates a
join that a reasonable person will write and that silently returns a number 101
times too large.

Which is why slide 20's *"more complex model design"* is worth taking literally.
The complexity of a galaxy schema is not in building it. It is in the fact that
**the model now contains a query that must not be written**, and nothing in the
schema stops it. That has to be handled somewhere: a semantic layer, a documented
pattern, or a view that pre-aggregates the safe combinations.

For this case study the recommendation is: build `fact_enrollment` as slide 29
specifies, add `fact_payment` only when detailed payment analysis is actually
needed, and when you do, ship the aggregate-then-join pattern alongside it.

### Question 10

Finally, make pandas check your assumption. Repeat question 7's join but add `validate="one_to_one"`. **This is supposed to fail.** Read the message and say what it actually verified.

In [ ]:
fact_enrollment = enr[["enrl_id", "course_id"]].copy()
fact_payment = (tx[["trans_id", "enrl_id"]]
                .merge(enr[["enrl_id", "course_id"]], on="enrl_id", how="left"))
print("distinct course_id in fact_enrollment:", fact_enrollment.course_id.nunique(),
      "over", len(fact_enrollment), "rows")
print("distinct course_id in fact_payment:   ", fact_payment.course_id.nunique(),
      "over", len(fact_payment), "rows")
print()
print(fact_enrollment.merge(fact_payment, on="course_id",
                            validate="one_to_one").shape)

```
distinct course_id in fact_enrollment: 24 over 2400 rows
distinct course_id in fact_payment:    24 over 4856 rows

MergeError: Merge keys are not unique in either left or right dataset; not a one-to-one merge.
Duplicates in left:
  course_id
       202
       208
       203
...
```

`validate="one_to_one"` checked the assumption and rejected it before producing a
single row.

Look at what the two lines above the error already told you: **24 distinct
`course_id` values across 2,400 rows on one side, and 24 across 4,856 on the
other.** Neither side is unique on the join key. That is the fan-out from question
7, visible in two `nunique()` calls, before any join runs.

`validate` takes four values and each is an assertion you are making out loud:

| value | you are claiming |
|---|---|
| `"one_to_one"` | the key is unique in **both** tables |
| `"one_to_many"` | unique on the left, repeated on the right |
| `"many_to_one"` | repeated on the left, unique on the right |
| `"many_to_many"` | no claim — the default behaviour |

The useful one in day-to-day work is **`many_to_one`**, on every join from a fact
table to a dimension. It asserts the dimension key is unique — which is exactly
the property that, when it silently fails, inflates a revenue total by a small
percentage nobody notices. One keyword argument, checked at run time, every time.

```python
fact.merge(dim_course, on="course_id", validate="many_to_one")
```

SQL has no direct equivalent, which is why the same guarantee is bought there
with a `PRIMARY KEY` or `UNIQUE` constraint on the dimension — and why Snowflake
**not enforcing** declared keys is worth remembering. Where the database will not
check, the load has to.

**What this sheet established:**

| | |
|---|---|
| star vs snowflake, storage | 318 chars vs 63 — **5.0x**, and irrelevant at this scale |
| star vs snowflake, joins | 1 vs 3, paid by every query forever |
| star vs snowflake, rename | 6 rows vs 1 — the real cost, and it is consistency, not bytes |
| both give | **identical answers**, including `Unknown` at 316 |
| galaxy, done wrong | 491,688 rows; revenue **101x** too large |
| galaxy, done right | aggregate each fact first, then join — both totals exact |

Worksheet 05 turns from the shape of the model to how it gets built: the
source-to-target mapping, and staging.